该笔记本会将以.tif格式存储的变量重新采样至数字高程模型（DEM）的分辨率（2米）水平。

导入相关模块

In [ ]:
from osgeo import gdal, gdalconst
import os

设置目录

In [ ]:
os.chdir(r"C:\Users\angelinkatula\Desktop\P8\climate_data")

需要进行重新采样的文件。（在此示例中，是针对未来气候情景RCP8.5的平均风速数据，时间范围为2700年至2100年）

In [ ]:
src_filename = 'wind_mean_future.tif'
src = gdal.Open(src_filename, gdalconst.GA_ReadOnly)
src_proj = src.GetProjection()
src_geotrans = src.GetGeoTransform()

检查其无数据值的情况

In [ ]:
NDV = src.GetRasterBand(1).GetNoDataValue()
NDV

1.0000000150474662e+30

所要求分辨率的文件

In [ ]:
# We want a section of source that matches this:
match_filename = r'C:\Users\angelinkatula\Desktop\P8\Variables\dem_elevation.tif'
match_ds = gdal.Open(match_filename, gdalconst.GA_ReadOnly)
match_proj = match_ds.GetProjection()
match_geotrans = match_ds.GetGeoTransform()
match_NDV = match_ds.GetRasterBand(1).GetNoDataValue()
wide = match_ds.RasterXSize
high = match_ds.RasterYSize
match_NDV

-9999.0

重新采样并将输出保存为.tif格式文件

In [ ]:
# Output / destination
dst_filename = r'wind_m_future_resampled_2m.tif'
dst = gdal.GetDriverByName('GTiff').Create(dst_filename, wide, high, 1, gdalconst.GDT_Float32)
dst.SetGeoTransform( match_geotrans )
dst.SetProjection( match_proj)
dst.GetRasterBand(1).SetNoDataValue(match_NDV)

# Do the work
gdal.ReprojectImage(src, dst, src_proj, match_proj, gdalconst.GRA_NearestNeighbour)
dst.FlushCache()
del dst # Flush